In [ ]:
#BACKPROPAGATION IN ANN

In [1]:
import numpy as np

# Activation
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Data
X = np.array([[0,1],
              [1,1],
              [1,0]])

y = np.array([[1],[0],[1]])

# Initialize weights
np.random.seed(1)
W1 = np.random.rand(2, 3)
W2 = np.random.rand(3, 1)

lr = 0.1

# Training
for epoch in range(1000):

    # Forward
    Z1 = np.dot(X, W1)
    A1 = sigmoid(Z1)

    Z2 = np.dot(A1, W2)
    A2 = sigmoid(Z2)

    # Loss (MSE)
    loss = np.mean((y - A2)**2)

    # Backprop
    dA2 = (A2 - y)
    dZ2 = dA2 * sigmoid_derivative(A2)
    dW2 = np.dot(A1.T, dZ2)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivative(A1)
    dW1 = np.dot(X.T, dZ1)

    # Update weights
    W1 -= lr * dW1
    W2 -= lr * dW2

# Final Output
print("Final Output:\n", A2)
print("Final Loss:", loss)

Final Output:
 [[0.67313323]
 [0.59394466]
 [0.66280413]]
Final Loss: 0.19110440093800699


In [ ]:
#BACKPROPAGATION IN CNN

In [2]:
import numpy as np

# -----------------------
# Activation Functions
# -----------------------
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# -----------------------
# Input (1 sample: 4x4 image)
# -----------------------
X = np.array([[1, 0, 1, 0],
              [0, 1, 0, 1],
              [1, 0, 1, 0],
              [0, 1, 0, 1]])

y = np.array([[1]])  # target

# -----------------------
# Initialize parameters
# -----------------------
np.random.seed(0)
F = np.random.randn(2, 2)      # 1 filter (2x2)
W = np.random.randn(9, 1)      # fully connected weights

lr = 0.01

# -----------------------
# Training
# -----------------------
for epoch in range(500):

    # ---------- FORWARD PASS ----------

    # Convolution (valid padding)
    conv_out = []
    for i in range(3):
        for j in range(3):
            region = X[i:i+2, j:j+2]
            conv_out.append(np.sum(region * F))

    conv_out = np.array(conv_out).reshape(1, -1)  # shape (1,9)

    # Activation
    A = relu(conv_out)

    # Fully Connected Layer
    Z = np.dot(A, W)
    output = sigmoid(Z)

    # Loss (Mean Squared Error)
    loss = np.mean((y - output) ** 2)

    # ---------- BACKPROPAGATION ----------

    # Output layer gradient
    d_output = (output - y)                          # dL/dO
    dZ = d_output * sigmoid_derivative(output)       # dL/dZ
    dW = np.dot(A.T, dZ)                             # gradient for FC weights

    # Backprop into convolution output
    dA = np.dot(dZ, W.T)                             # shape (1,9)
    d_conv = dA * relu_derivative(A)

    # Filter gradient
    dF = np.zeros_like(F)
    idx = 0
    for i in range(3):
        for j in range(3):
            region = X[i:i+2, j:j+2]
            dF += d_conv[0, idx] * region
            idx += 1

    # ---------- UPDATE ----------
    W -= lr * dW
    F -= lr * dF

    # Print every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")

# -----------------------
# Final Output
# -----------------------
print("\nFinal Output:", output)
print("Final Loss:", loss)
print("\nUpdated Filter:\n", F)
print("\nUpdated FC Weights:\n", W)

Epoch 0, Loss: 3.366696937661958e-14
Epoch 100, Loss: 3.366696937661958e-14
Epoch 200, Loss: 3.366696937661958e-14
Epoch 300, Loss: 3.366696937661958e-14
Epoch 400, Loss: 3.366696937661958e-14

Final Output: [[0.99999982]]
Final Loss: 3.366696937661958e-14

Updated Filter:
 [[1.76405235 0.40015721]
 [0.97873798 2.2408932 ]]

Updated FC Weights:
 [[ 1.86755799]
 [-0.97727788]
 [ 0.95008842]
 [-0.15135721]
 [-0.10321885]
 [ 0.4105985 ]
 [ 0.14404357]
 [ 1.45427351]
 [ 0.76103773]]


In [ ]:
#BACKPROPAGATION IN RNN

In [3]:
import numpy as np

# Activation functions
def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - x**2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Input sequence (2 timesteps, 2 features)
X = [np.array([[1],[0]]), 
     np.array([[0],[1]])]

# Target output
y = np.array([[1]])

# Initialize weights
np.random.seed(1)
Wx = np.random.randn(2, 2)   # input → hidden
Wh = np.random.randn(2, 2)   # hidden → hidden
Wy = np.random.randn(2, 1)   # hidden → output

# Initial hidden state
h_prev = np.zeros((2,1))

learning_rate = 0.01
epochs = 500

for epoch in range(epochs):

    # --------------------
    # FORWARD PASS
    # --------------------
    hs = []
    h = h_prev

    for t in range(len(X)):
        h = tanh(np.dot(Wx, X[t]) + np.dot(Wh, h))
        hs.append(h)

    # Final output
    y_pred = sigmoid(np.dot(Wy.T, h))

    # --------------------
    # LOSS (MSE)
    # --------------------
    loss = np.mean((y - y_pred)**2)

    # --------------------
    # BACKPROPAGATION (BPTT)
    # --------------------
    dWy = (y_pred - y) * hs[-1]   # output layer gradient

    dWx = np.zeros_like(Wx)
    dWh = np.zeros_like(Wh)

    dh_next = np.zeros_like(hs[0])

    # Backward through time
    for t in reversed(range(len(X))):
        
        dh = np.dot(Wy, (y_pred - y)) + dh_next
        
        dtanh = dh * tanh_derivative(hs[t])
        
        dWx += np.dot(dtanh, X[t].T)
        
        if t > 0:
            dWh += np.dot(dtanh, hs[t-1].T)
        
        dh_next = np.dot(Wh.T, dtanh)

    # --------------------
    # WEIGHT UPDATE
    # --------------------
    Wx -= learning_rate * dWx
    Wh -= learning_rate * dWh
    Wy -= learning_rate * dWy

# --------------------
# FINAL OUTPUT
# --------------------
print("Final Prediction:", y_pred)
print("Final Loss:", loss)

print("\nUpdated Weights:")
print("Wx:\n", Wx)
print("Wh:\n", Wh)
print("Wy:\n", Wy)

Final Prediction: [[0.87518393]]
Final Loss: 0.015579051092870371

Updated Weights:
Wx:
 [[ 1.79213442 -0.46651948]
 [-0.64299396 -0.98573611]]
Wh:
 [[ 1.00112963 -2.38248443]
 [ 1.82701319 -0.81270849]]
Wy:
 [[1.45205917]
 [0.68331635]]
